# Attacks on the dataset belonging to this paper

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import trange, tqdm
import seaborn as sns
import bokeh.plotting as bk
import lpips
import librosa
import torchaudio
import torch
import math
from scipy.signal import find_peaks

from skimage.metrics import structural_similarity as ssim
import torch.nn.functional as F


from bokeh.io import output_notebook 
output_notebook()

## Correlation based attack

In [2]:
def correlation(target_trace, template_trace, poi_start, poi_end):
    corr_array = []
    window = poi_end-poi_start
    template_trace_sliced = template_trace[poi_start:poi_end, ]
    for timesample in trange(0, len(target_trace)-window):
        corr = np.corrcoef(target_trace[timesample:timesample+window], template_trace_sliced)[0,1] # Take the non-identity part of the correlation matrix. Note that [0,1] = [1,0]
        corr_array.append(corr)
    return np.array(corr_array)

In [3]:
def plot_similarity(similarity_arrays, scalar, left, right, method):
    plt.figure(figsize=(10,5))
    plt.plot(similarity_arrays[0], label = "bit 0")
    plt.plot(similarity_arrays[1], label = "bit 1")
    plt.axvspan(left, right, alpha=0.2)
    plt.title(f"Correlation comparison for {scalar << 1}P and {(scalar << 1) | 1}P using {method.__name__}")
    plt.xlabel("Timesamples")
    plt.ylabel("Delta power consumption")
    plt.legend()
    plt.show()
    
def compute_correlation(target_trace, template_trace):
    corr = np.corrcoef(target_trace, template_trace)[0,1] # Take the non-identity part of the correlation matrix. Note that [0,1] = [1,0]
    return np.abs(corr)

def pattern_match(target_trace, template_trace, comparison_function, poi_start, poi_end, step=1):
    comparison_array = []
    window = poi_end - poi_start
    template_trace_sliced = template_trace[poi_start:poi_end]
    for timesample in trange(0, len(target_trace) - window, step):
        comparison = comparison_function(target_trace[timesample:timesample + window], template_trace_sliced)
        comparison_array.append(comparison)
    return np.array(comparison_array)

def get_window(any_template, iteration, step_size=1, height=0.7, distance=20000, prominence=0.15):
    #signal = pattern_match(target_trace, any_template, compute_correlation, poi_start, poi_end)
    peaks, _ = find_peaks(any_template, height=height, distance=distance, prominence=prominence)
    left = peaks[iteration+1] - (distance // 2)
    right = peaks[iteration+1] + (distance // 2)
    
    return left//step_size, right//step_size

def determine_keybit(comparison_arrays, window_start, window_end, scalar, method):
    # Decide scoring rule
    if method == "compute_distance_lpips":
        reducer = np.min
        better = lambda a, b: a < b
        label = "Min"
    else:
        reducer = np.max
        better = lambda a, b: a > b
        label = "Max"

    peak_scores = []

    for idx, comp_array in enumerate(comparison_arrays):

        if window_start >= len(comp_array):
            selected_peak = 0
        else:
            end = min(window_end, len(comp_array))
            selected_peak = reducer(comp_array[window_start:end])

        peak_scores.append(selected_peak)

        print(f"Template {idx}")
        print("  Window:", window_start, "-", window_end)
        print(f"  {label} in window:", selected_peak)

    score_bit0, score_bit1 = peak_scores

    print("Score bit 0:", score_bit0)
    print("Score bit 1:", score_bit1)

    scalar_bit = 1 if better(score_bit1, score_bit0) else 0
    scalar = (scalar << 1) | scalar_bit

    return scalar_bit, scalar


## Load in dataset

In [ ]:
traces = np.load("../../datasets/ota3/ecc-scalarmult-montgomery.npz")["traces"]
target_trace = traces[0]
template_traces = traces[1:]
nr_samples = len(target_trace)
print(template_traces)

print(nr_samples)

## Correlation based attack

In [ ]:
# Do the sequential key bit recovery

keybits = 6
poi_start = 10000 # POI is starting from first ladderstep + cswap.
poi_end =  37900

recovered_key = [0, 1] # MSB is always 1, add it already to the array
scalar = 1

for iteration, keybit_to_guess in enumerate(range(0, keybits*2, 2)):

    # Determine input points based on previous key bits
    # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
    
    # Collect traces
    # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

    # Create templates from the collected traces
    template0_targetbit = template_traces[2*iteration, :]
    template1_targetbit = template_traces[2*iteration + 1, :]
    templates_targetbit = [template0_targetbit, template1_targetbit]

    comparison_arrays = []
    for template in templates_targetbit:
        comparison_array = np.abs(pattern_match(target_trace, template, compute_correlation, poi_start, poi_end, step=1))
        comparison_arrays.append(comparison_array)
    
    # Perform pattern matching with templates to target trace based on Correlation
    window_start, window_end = get_window(comparison_arrays[0], iteration)
    
    plot_similarity(comparison_arrays, scalar, window_start, window_end, compute_correlation)
    
    # Determine the key bit based on the pattern matching results
    best_keyguess, scalar = determine_keybit(comparison_arrays, window_start, window_end, scalar, compute_correlation)

    recovered_key.append(best_keyguess)
    print("Recovered key bit:", best_keyguess)
    print("Current scalar:", scalar)


print("The found key: " + str(recovered_key))

## Similarity based attack (SSIM + LPIPS)

In [48]:
def calc_spectrogram(trace, n_fft = 512): #NOTE: it is very sensitive for this value
    trace_tensor = torch.tensor(trace, dtype=torch.float32)
    spec = torchaudio.transforms.Spectrogram(n_fft=n_fft)(trace_tensor)
    spec_db = librosa.power_to_db(spec.numpy(), ref=np.max)

    # Normalize to [0,1]
    spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)

    return spec_db

def spec_to_tensor(spec_db, target_size):
    # Convert to 3-channel float32 tensor for LPIPS: (C, H, W)
    spec_tensor = torch.tensor(spec_db).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    # Resize to target_size
    spec_resized = F.interpolate(spec_tensor, size=target_size, mode='bilinear', align_corners=False)

    return spec_resized

In [ ]:
## LPIPS model
device = "cuda" if torch.cuda.is_available() else "cpu" 
lpips_model = lpips.LPIPS(net='alex')
lpips_model.eval()
lpips_model.to(device)

In [50]:
def compute_distance_lpips(target_trace, template_trace):
    tensor_size = (224, 224)  # LPIPS expects 224x224 input
    
    target_spec  = calc_spectrogram(target_trace)
    template_spec = calc_spectrogram(template_trace)

    target_tensor = spec_to_tensor(target_spec, tensor_size).to(device)
    template_tensor = spec_to_tensor(template_spec, tensor_size).to(device)

    d = lpips_model.forward(target_tensor,template_tensor)
    return d.detach().numpy().ravel()[0]

def compute_structural_similarity(target_trace, template_trace):
    """Compute structural similarity between two traces."""
    target_tensor = calc_spectrogram(target_trace, 512)
    template_tensor = calc_spectrogram(template_trace, 512)

    return ssim(target_tensor, template_tensor, data_range=target_tensor.max() - target_tensor.min())

In [ ]:
trace1 = np.random.randn(nr_samples).astype(np.float32)
trace2 = np.random.randn(nr_samples).astype(np.float32)

similarity = compute_structural_similarity(trace1, trace2)
print(f"Structural Similarity (SSIM): {similarity:.4f}")

distance = compute_distance_lpips(trace1, trace2)
print(f"LPIPS distance: {distance:.4f}")

In [ ]:
# Do the sequential key bit recovery

keybits = 6
scalar = 1
similarity_methods = [compute_structural_similarity, compute_distance_lpips]
recovered_key = np.ones(shape=(len(similarity_methods), keybits + 2))  # Initialize with ones to set the first bit for each method to 1
recovered_key[:, 0] = 0  # Set the MSB to 1 for all methods

poi_start = 10000
poi_end = 37900
step_size = 1


# Perform pattern matching with templates to target trace based on Correlation
for j, method in enumerate(similarity_methods):
    scalar = 1
    for iteration, keybit_to_guess in enumerate(range(0, keybits*2, 2)):

        # Determine input points based on previous key bits
        # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
        
        # Collect traces
        # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

        # Create templates from the collected traces
        template0_targetbit = template_traces[2*iteration, :]
        template1_targetbit = template_traces[2*iteration + 1, :]
        templates_targetbit = [template0_targetbit, template1_targetbit]

        # Find peaks automatically
        corr_array = np.abs(pattern_match(target_trace, templates_targetbit[0], compute_correlation, poi_start, poi_end, step=1))
        window_start, window_end = get_window(corr_array, iteration, step_size=step_size, height=0.7, distance=20000, prominence=0.15)

        comparison_arrays = []
        for template in templates_targetbit:
            comparison_array = pattern_match(target_trace, template, method, poi_start, poi_end, step=step_size)
            comparison_arrays.append(comparison_array)

        plot_similarity(comparison_arrays, scalar, window_start, window_end, method)
        best_keyguess, scalar = determine_keybit(comparison_arrays, window_start, window_end, scalar, method)
        
        recovered_key[j, iteration+1] = best_keyguess
        print("Best key guess for method {}: {}".format(method.__name__, best_keyguess))
        print("Scalar value for method {}: {}".format(method.__name__, scalar))

print("The correct key: 01101010")

for j, method in enumerate(similarity_methods):
    print(f"Recovered key using {method.__name__}: " + str(recovered_key[j]))

    